In [1]:
import numpy as np
import lmdb
import pickle

root = '/data/protein/SKData/DenoisingData/pcq'
MOL_LST = lmdb.open(f'{root}/MOL_LMDB', readonly=True, subdir=True, lock=False)

# get the MOL_LST length
with MOL_LST.begin() as txn:
    _keys = list(txn.cursor().iternext(values=False))
    mol_len = len(_keys)



from rdkit import Chem
import copy
from rdkit.Chem import AllChem
# from rdkit.Chem.Draw import IPythonConsole
# IPythonConsole.ipython_useSVG = True

import multiprocessing
from tqdm import tqdm

In [13]:
rdkit_failed_cnt = 0
rdkit_conf_mol_lst = []

for ky in _keys:
    serialized_data = MOL_LST.begin().get(ky)
    mol = pickle.loads(serialized_data)
    test_mol = copy.deepcopy(mol)
    # cids = AllChem.EmbedMultipleConfs(test_mol, numConfs=1, numThreads=0)
    try:
        test_mol.RemoveConformer(0)
        cids = AllChem.EmbedMultipleConfs(test_mol, numConfs=10, numThreads=16, pruneRmsThresh=0.1, maxAttempts=50, useRandomCoords=True, randomSeed=42)
        

        if len(cids) < 1:
            rdkit_failed_cnt += 1
            print('rdkit generate fail')
        else:
            AllChem.MMFFOptimizeMoleculeConfs(test_mol, numThreads=8)
    except Exception as e:
        print(f'exeption captured {e}')
    rdkit_conf_mol_lst.append(test_mol)
    break

In [14]:
# save the test_mol to sdf file
from rdkit.Chem import SDWriter
writer = SDWriter('test.sdf')
for mol in rdkit_conf_mol_lst:
    for cid in range(mol.GetNumConformers()):
        writer.write(mol, confId=cid)

In [15]:
mol.GetNumConformers()

5